# Mandarin (en-zh) fine-tune on Colab GPU

Before running: **Runtime -> Change runtime type -> Hardware accelerator -> GPU** (T4 or whatever's available) -> Save.

No GitHub access needed -- `train.py` has no local imports, so this notebook
writes it and its config directly to the Colab VM instead of cloning a repo.

**One-time manual step, do this first**: upload the 4 processed data files
(`data/processed/en-zh/{train,val}.{en,zh}`, ~22MB total, from the local repo)
to a Google Drive folder, e.g. `MyDrive/medical_vlm_data/en-zh/`. Update
`DRIVE_DATA_DIR` below to match wherever you put them.

Checkpoints are written to Drive (not the ephemeral Colab VM disk), so if
the session disconnects (free tier: ~90 min idle / ~12h max), just
reconnect and re-run this notebook top to bottom -- the last cell
auto-detects an existing checkpoint and resumes from it instead of
starting over. Every cell that depends on the working directory sets it
explicitly, so re-running just a subset of cells after a disconnect (not
strictly top-to-bottom) also works.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- stop and pick a GPU runtime')
assert torch.cuda.is_available(), 'No GPU attached -- use Runtime -> Change runtime type -> GPU'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

In [ ]:
import os

# No GitHub needed: train.py has zero local imports (only external packages),
# so the next two cells just write it and the config straight to disk instead
# of cloning a repo -- avoids the private-repo auth problem entirely.
REPO_DIR = "/content/repo"
os.makedirs(f"{REPO_DIR}/src", exist_ok=True)
os.makedirs(f"{REPO_DIR}/configs", exist_ok=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
%%writefile src/train.py
"""Fine-tune a pretrained MarianMT checkpoint on a medical parallel corpus.

Mirrors teammate 2's per-direction MarianMT fine-tuning pattern: one pretrained
Helsinki-NLP/opus-mt-* checkpoint in, one fine-tuned checkpoint out, driven by
a small YAML config so the same script covers every direction (en-zh, zh-en,
en-hi, hi-en, ...) with only the config changed.

Usage:
    python src/train.py --config configs/en-zh.yaml
"""
import argparse
from pathlib import Path

import numpy as np
import torch
import yaml
from datasets import Dataset
from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)


def load_parallel_file_pair(src_path: Path, tgt_path: Path) -> Dataset:
    with src_path.open(encoding="utf-8") as f:
        src_lines = [l.rstrip("\n") for l in f]
    with tgt_path.open(encoding="utf-8") as f:
        tgt_lines = [l.rstrip("\n") for l in f]
    assert len(src_lines) == len(tgt_lines), (
        f"{src_path} has {len(src_lines)} lines, {tgt_path} has {len(tgt_lines)}"
    )
    return Dataset.from_dict({"src": src_lines, "tgt": tgt_lines})


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--config", required=True, type=Path)
    ap.add_argument("--resume-from-checkpoint", default=None)
    args = ap.parse_args()

    cfg = yaml.safe_load(args.config.read_text(encoding="utf-8"))

    base_model = cfg["base_model"]
    output_dir = Path(cfg["output_dir"])
    max_src_len = cfg.get("max_source_length", 128)
    max_tgt_len = cfg.get("max_target_length", 128)

    print(f"Loading base model {base_model} ...")
    tokenizer = MarianTokenizer.from_pretrained(base_model)
    model = MarianMTModel.from_pretrained(base_model)

    train_ds = load_parallel_file_pair(Path(cfg["train_src"]), Path(cfg["train_tgt"]))
    val_ds = load_parallel_file_pair(Path(cfg["val_src"]), Path(cfg["val_tgt"]))
    print(f"Train examples: {len(train_ds)} | Val examples: {len(val_ds)}")

    def preprocess(batch):
        model_inputs = tokenizer(
            batch["src"], max_length=max_src_len, truncation=True,
        )
        labels = tokenizer(
            text_target=batch["tgt"], max_length=max_tgt_len, truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    train_tok = train_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])
    val_tok = val_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    def compute_metrics(eval_preds):
        import sacrebleu

        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
        return {"bleu": bleu.score}

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        learning_rate=cfg.get("learning_rate", 2e-5),
        per_device_train_batch_size=cfg.get("per_device_train_batch_size", 16),
        per_device_eval_batch_size=cfg.get("per_device_eval_batch_size", 16),
        weight_decay=cfg.get("weight_decay", 0.01),
        num_train_epochs=cfg.get("num_train_epochs", 3),
        predict_with_generate=True,
        generation_max_length=max_tgt_len,
        fp16=torch.cuda.is_available(),
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        save_total_limit=cfg.get("save_total_limit", 2),
        report_to=cfg.get("report_to", []),
        group_by_length=cfg.get("group_by_length", False),
        dataloader_num_workers=cfg.get("dataloader_num_workers", 0),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.get("early_stopping_patience", 3))],
    )

    trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)

    final_dir = output_dir / "final"
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    print(f"Saved fine-tuned model to {final_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile configs/en-zh-colab-gpu.yaml
# Same fine-tune as configs/en-zh.yaml, retuned for GPU throughput (Colab).
# Kept separate from the CPU config on purpose: batch size and dataloader
# settings are hardware-dependent tuning, not methodology, so the CPU run's
# config stays untouched. learning_rate is deliberately left unchanged from
# the CPU config, even though batch size is 4x larger -- scaling both at once
# would conflate a throughput change with an optimization-trajectory change,
# and this is a small enough model/LR that the standard linear-scaling
# argument for bumping LR with batch size doesn't clearly apply here.
#
# Data expected at data/processed/en-zh/{train,val}.{en,zh}
# (same pooled data as the CPU run -- copy from Drive, see colab_train_en_zh.ipynb)

base_model: Helsinki-NLP/opus-mt-en-zh
output_dir: models/opus-mt-en-zh-medical

train_src: data/processed/en-zh/train.en
train_tgt: data/processed/en-zh/train.zh
val_src: data/processed/en-zh/val.en
val_tgt: data/processed/en-zh/val.zh

max_source_length: 128
max_target_length: 128

num_train_epochs: 3
learning_rate: 2.0e-5

# 4x the CPU config's batch size -- this is a ~77M-param model at seq_len 128,
# comfortably small for any Colab GPU tier (T4 and up). If `!nvidia-smi` in
# the notebook shows headroom, this can go higher (e.g. 128) for more speed.
per_device_train_batch_size: 64
per_device_eval_batch_size: 64

# Buckets similar-length examples into the same batch, so batches pad to the
# length of their own examples instead of the longest in a random batch --
# real speedup (less wasted compute on padding tokens) with no quality cost.
group_by_length: true

# Colab typically gives ~2 CPU cores; overlaps batch collation with GPU compute
# instead of blocking on it (irrelevant on the CPU run, where dataloading was
# never the bottleneck -- GPU compute is fast enough that it can be).
dataloader_num_workers: 2

weight_decay: 0.01
early_stopping_patience: 3
save_total_limit: 2
report_to: []

In [ ]:
# Deliberately NOT reinstalling torch -- Colab's preinstalled torch already has
# CUDA wired up correctly; reinstalling from requirements.txt risks silently
# swapping it for a CPU-only build.
!pip install -q transformers sentencepiece sacrebleu PyYAML accelerate sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

os.chdir("/content/repo")  # hardcoded so this cell is self-contained even after a partial re-run

DRIVE_DATA_DIR = "/content/drive/MyDrive/medical_vlm_data/en-zh"  # where you uploaded train/val.{en,zh}
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-zh-medical"

assert os.path.isdir(DRIVE_DATA_DIR), f"{DRIVE_DATA_DIR} not found -- upload the processed data there first"
for f in ["train.en", "train.zh", "val.en", "val.zh"]:
    assert os.path.isfile(os.path.join(DRIVE_DATA_DIR, f)), f"missing {f} in {DRIVE_DATA_DIR}"

os.makedirs("data/processed/en-zh", exist_ok=True)
for f in ["train.en", "train.zh", "val.en", "val.zh"]:
    shutil.copy(os.path.join(DRIVE_DATA_DIR, f), os.path.join("data/processed/en-zh", f))

# models/ -> Drive, so checkpoints (and the final model) survive a disconnect
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
if os.path.islink("models") or os.path.isdir("models"):
    if os.path.islink("models"):
        os.remove("models")
    else:
        shutil.rmtree("models")
os.symlink("/content/drive/MyDrive/medical_vlm_checkpoints", "models")
print("Data and checkpoint dir ready. cwd:", os.getcwd())

In [ ]:
import os, glob

os.chdir("/content/repo")  # hardcoded so this cell is self-contained even after a partial re-run

DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-zh-medical"
checkpoints = sorted(glob.glob(os.path.join(DRIVE_CHECKPOINT_DIR, "checkpoint-*")), key=os.path.getmtime)
resume_flag = f'--resume-from-checkpoint "{checkpoints[-1]}"' if checkpoints else ""
if checkpoints:
    print(f"Resuming from {checkpoints[-1]}")
else:
    print("No existing checkpoint found -- starting fresh.")

# en-zh-colab-gpu.yaml, not en-zh.yaml: same data/model/epochs, but batch size,
# group_by_length, and dataloader_num_workers are retuned for GPU throughput
# (see the config file's comments for why each change is safe).
!python src/train.py --config configs/en-zh-colab-gpu.yaml {resume_flag}